# 3 - Metadaten abholen

Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen. Dazu wird ein Unterordner {signature} im Ordner 'metadata' erstellt. 

### Metadaten aus Alma (SRU, marcxml)

Mit der MMS ID werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/{signature}/signature.xml. 


### Datacite - Dublin Core (OAI, XML)

Daten aus Zenodo-Repositories werden mit Datacite-Metadaten ins Archiv eingelagert.
Dazu wird die Zenodo-ID und das OAI-Set benötigt.
Rate Limiting Daten für Zenodo (OAI und API): https://about.zenodo.org/principles/ Punkt 11 (fast zuunterst)


### Weitere Metadaten

z.B. OCR, etc. (TODO)

In [5]:
import requests
import json
from datetime import datetime
from pathlib import Path
from sickle import Sickle
import config
import time


# which metadata is available:
marc = config.marcxml
marc_url = config.marcxml_baseurl

datacite = config.datacite
dublincore = config.dc
zenodo_url = config.zenodo_baseurl
zenodo_set = config.zenodo_set

# general config:

urn = config.ingest_workflow
collection = config.collection_id

md_path = f'{collection}/{config.metadata_path}'
org_id = config.organisation_id

input_file = f'{config.inventory_file}'
counter = 0

with open(input_file, encoding="utf-8") as data_file:    
    data = json.load(data_file)
    for value in data:
        
        counter+= 1 
        
        if counter<= 15: # debug mode, for prod: replace with if True:

            # create new directory for each signature (ignore, if it already exists)            
            foldername = value["references"][-1]
            Path(f'{md_path}/{foldername}').mkdir(parents=True, exist_ok=True)   

            identifiers = {}
            for item in value["identifiers"]:
                # split identifiers in dict
                [key, value] = item.split(':',1)
                identifiers[key] = value          

            # marcxml data:        
            if marc == 'True':     

                # get mms_id
                mmsid = identifiers['mmsid']

                # get SRU response
                query = marc_url+mmsid
                response = requests.get(query)
                if response.status_code != 200:
                    raise Exception(f"SRU request failed with status code {response.status_code}")

                # Save the response content as xml to a new directory
                marcxmlfile = f"{md_path}/{foldername}/{mmsid}.xml"

                with open(marcxmlfile, 'wb') as file:
                    file.write(response.content)
                    print(f"\nMarc Record #{counter} with ID {mmsid} downloaded.")


            # datacite metadata        
            if (datacite == "True"):       

                # get zenodo id and start OAI-PMH request
                zenodo_id = identifiers['zenodo']
                sickle = Sickle(zenodo_url)            
                datacite_response = sickle.GetRecord(identifier=f"oai:zenodo.org:{zenodo_id}", metadataPrefix='oai_datacite')
                datacitefile = f'{md_path}/{foldername}/{zenodo_id}_datacite.xml'

                with open(datacitefile, 'w', encoding="utf-8") as file:
                    file.write(datacite_response.raw)
                    print(f'#{counter} - {zenodo_id}: Datacite Metadata downloaded. ')

                # wait 1 second every 10 records so as not to overshoot zenodo rate limiting. 
                if counter%10 == 0: time.sleep(1)

            # dublincore metadata        
            if (dublincore == "True"):  

                # get zenodo id and start OAI-PMH request
                zenodo_id = identifiers['zenodo']
                sickle = Sickle(zenodo_url)            
                dc_response = sickle.GetRecord(identifier=f"oai:zenodo.org:{zenodo_id}", metadataPrefix='oai_dc')
                dc_file = f'{md_path}/{foldername}/{zenodo_id}_dc.xml'

                with open(dc_file, 'w', encoding="utf-8") as file:
                    file.write(dc_response.raw)
                    print(f'#{counter} - {zenodo_id}: DC Metadata downloaded. ')

                # wait 1 second every 10 records so as not to overshoot zenodo rate limiting. 
                if counter%10 == 0: time.sleep(1)                

    
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

#1 - 10466776: Datacite Metadata downloaded. 
#1 - 10466776: DC Metadata downloaded. 
#2 - 10471407: Datacite Metadata downloaded. 
#2 - 10471407: DC Metadata downloaded. 
#3 - 10471212: Datacite Metadata downloaded. 
#3 - 10471212: DC Metadata downloaded. 
#4 - 10471210: Datacite Metadata downloaded. 
#4 - 10471210: DC Metadata downloaded. 
#5 - 10471153: Datacite Metadata downloaded. 
#5 - 10471153: DC Metadata downloaded. 
#6 - 10470760: Datacite Metadata downloaded. 
#6 - 10470760: DC Metadata downloaded. 
#7 - 10470627: Datacite Metadata downloaded. 
#7 - 10470627: DC Metadata downloaded. 
#8 - 10469478: Datacite Metadata downloaded. 
#8 - 10469478: DC Metadata downloaded. 
#9 - 10468315: Datacite Metadata downloaded. 
#9 - 10468315: DC Metadata downloaded. 
#10 - 10137907: Datacite Metadata downloaded. 
#10 - 10137907: DC Metadata downloaded. 
#11 - 10407260: Datacite Metadata downloaded. 
#11 - 10407260: DC Metadata downloaded. 
#12 - 10407165: Datacite Metadata downloaded. 
#12